# 02 — Preprocessing


Goals:
- Define predictors and target
- Decide which features to retain/exclude
- Separate numeric and categorical columns
- Create train/test split
- Build preprocessing pipeline


## 1. Library imports

Import the required Python libraries for data manipulation, dataset splitting,
numerical scaling, categorical encoding, missing-value imputation, and building
a reusable preprocessing pipeline.

In [124]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# Project constants
TARGET_COLUMN = "y"
RANDOM_STATE = 42

#Added to reuse preprocessing from src
from pathlib import Path
import sys

# Identify project root
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

# Add project root to Python path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import reusable preprocessing function
from src.preprocessing import build_preprocessor

from scipy.sparse import issparse

## 2. Data load and Data Exploration

Load the Bank Marketing dataset (`bank-full.csv`) and perform an initial
exploration to understand its structure i.e. No. of observations, input features, and target.

In [125]:
# Load dataset
df = pd.read_csv(PROJECT_ROOT / "data" / "bank-full.csv", sep=";")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (45211, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


## 3. Define Predictors and Target

Define the target variable (`y`) and encode into binary values (yes = 1, no = 0) and select the predictor variables (`X`).
Exclude five features (`default`, `day`, `duration`, `pdays`, and `poutcome`), retaining 11 predictors for model development.


## Feature Inclusion and Exclusion Decisions

The Bank Marketing dataset contains 16 input features and one target variable (`y`). For this project, 11 predictors are selected, and 5 are excluded.

**Included Features (11):**
- Numerical: `age`, `balance`, `campaign`, `previous`
- Categorical: `job`, `marital`, `education`, `housing`, `loan`, `contact`, `month`

**Excluded Features (5):**
- `duration`: Excluded to prevent data leakage because call duration is only known after the call.
- `default`: Excluded because the feature is highly imbalanced and may provide limited predictive information.
- `day`: Excluded to avoid relying on potentially unstable campaign scheduling patterns.
- `pdays`: Excluded from the initial model because its special value (`-1`) requires additional handling.
- `poutcome`: Excluded to simplify the initial model, although previous campaign outcomes may contain useful predictive information.

These are preliminary feature-selection decisions. Their impact on model performance has not yet been evaluated and can be revisited during model evaluation.

The selected features will be used to predict whether a customer subscribes to a term deposit (`y = 1`) or not (`y = 0`).

In [126]:
# Define target variable
y = df[TARGET_COLUMN].map({"yes": 1, "no": 0})

# Exclude selected features
features_to_exclude = [
    "default", "day", "duration", "pdays", "poutcome"
]

# Define predictors
X = df.drop(columns=[TARGET_COLUMN] + features_to_exclude)

print("\nSelected predictors:", X.columns.tolist())


print("\nTarget distribution:")
print(y.value_counts())

print("\nTarget distribution (%):")
print(y.value_counts(normalize=True) * 100)


Selected predictors: ['age', 'job', 'marital', 'education', 'balance', 'housing', 'loan', 'contact', 'month', 'campaign', 'previous']

Target distribution:
y
0    39922
1     5289
Name: count, dtype: int64

Target distribution (%):
y
0    88.30152
1    11.69848
Name: proportion, dtype: float64


## 4. Identify Numerical and Categorical Features

Identify numerical and categorical predictors to apply appropriate preprocessing techniques: scaling for numerical features and encoding (one-hot) for categorical features.

In [127]:
# Identify numerical features
numerical_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

# Identify categorical features
categorical_features = X.select_dtypes(
    include=["object", "str", "category"]
).columns.tolist()

print("Numerical Features:", numerical_features)
print("Categorical Features:", categorical_features)

print("\nNumber of numerical features:", len(numerical_features))
print("Number of categorical features:", len(categorical_features))

Numerical Features: ['age', 'balance', 'campaign', 'previous']
Categorical Features: ['job', 'marital', 'education', 'housing', 'loan', 'contact', 'month']

Number of numerical features: 4
Number of categorical features: 7


## 5. Data Quality Checks and Handling Missing and Unknown Values

Check for actual missing values (NaN) and "unknown" entries in the selected predictors. Retain "unknown" as a valid categorical value and handle actual missing values using imputation in the preprocessing pipeline. Also, check any invalid numerical values, and duplicate records.

In [128]:
# Check missing values
print("Missing values:")
print(X.isnull().sum())

# Check unknown values in categorical features
print("\nUnknown values:")
print((X[categorical_features] == "unknown").sum())

# Check numerical validity
print("\nInfinite numerical values:")
print(np.isinf(X[numerical_features].to_numpy(dtype=float)).sum())

# Check duplicate records
print("\nDuplicate records:", df.duplicated().sum())

Missing values:
age          0
job          0
marital      0
education    0
balance      0
housing      0
loan         0
contact      0
month        0
campaign     0
previous     0
dtype: int64

Unknown values:
job            288
marital          0
education     1857
housing          0
loan             0
contact      13020
month            0
dtype: int64

Infinite numerical values:
0

Duplicate records: 0


## 6. Train/Test/Validation Split

Split the dataset into 70% training, 15% testing, and 15% validation sets using stratified sampling.

Stratified sampling preserves the original target class distribution (88.3% no and 11.7% yes) across all three sets, ensuring balanced representation of the original class proportions.

In [129]:
# Split dataset into 70% training and 30% temporary data
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y
)

# Split temporary data equally into testing and validation sets
X_test, X_val, y_test, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Validation features:", X_val.shape)

print("\nTraining target:", y_train.shape)
print("Testing target:", y_test.shape)
print("Validation target:", y_val.shape)

print("Original:", y.value_counts(normalize=True).round(3).to_dict())
print("Training:", y_train.value_counts(normalize=True).round(3).to_dict())
print("Validation:", y_val.value_counts(normalize=True).round(3).to_dict())
print("Testing:", y_test.value_counts(normalize=True).round(3).to_dict())

Training features: (31647, 11)
Testing features: (6782, 11)
Validation features: (6782, 11)

Training target: (31647,)
Testing target: (6782,)
Validation target: (6782,)
Original: {0: 0.883, 1: 0.117}
Training: {0: 0.883, 1: 0.117}
Validation: {0: 0.883, 1: 0.117}
Testing: {0: 0.883, 1: 0.117}


## 7. Build Preprocessing Pipeline

Create preprocessing pipelines to handle missing values, standardize numerical features, and encode categorical features.

Fit the preprocessing pipeline only on the training data and apply the same transformations to the validation and testing sets to prevent data leakage.

In [130]:
# Create reusable preprocessing pipeline
preprocessor = build_preprocessor(
    numerical_features,
    categorical_features
)

# Fit only on training data
X_train_processed = preprocessor.fit_transform(X_train)

# Transform validation and testing data
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

# Verify transformed dimensions
print("Training shape:", X_train_processed.shape)
print("Validation shape:", X_val_processed.shape)
print("Testing shape:", X_test_processed.shape)

# Inspect generated feature names
print("\nGenerated feature names:")
print(preprocessor.get_feature_names_out())

Training shape: (31647, 42)
Validation shape: (6782, 42)
Testing shape: (6782, 42)

Generated feature names:
['numeric__age' 'numeric__balance' 'numeric__campaign' 'numeric__previous'
 'categorical__job_admin.' 'categorical__job_blue-collar'
 'categorical__job_entrepreneur' 'categorical__job_housemaid'
 'categorical__job_management' 'categorical__job_retired'
 'categorical__job_self-employed' 'categorical__job_services'
 'categorical__job_student' 'categorical__job_technician'
 'categorical__job_unemployed' 'categorical__job_unknown'
 'categorical__marital_divorced' 'categorical__marital_married'
 'categorical__marital_single' 'categorical__education_primary'
 'categorical__education_secondary' 'categorical__education_tertiary'
 'categorical__education_unknown' 'categorical__housing_no'
 'categorical__housing_yes' 'categorical__loan_no' 'categorical__loan_yes'
 'categorical__contact_cellular' 'categorical__contact_telephone'
 'categorical__contact_unknown' 'categorical__month_apr'
 'ca

## 8. Validate and Save Preprocessing Results

Validate the preprocessed training, validation, and testing datasets by checking their dimensions and ensuring there are no missing or infinite values.

Save the fitted preprocessing pipeline for reuse in model training and future predictions.

In [131]:
# Validate processed datasets
for name, data in [
    ("Training", X_train_processed),
    ("Validation", X_val_processed),
    ("Testing", X_test_processed)
]:
    # Convert sparse matrices to dense arrays for validation
    if issparse(data):
        data = data.toarray()

    data = np.asarray(data, dtype=np.float64)

    print(f"{name} shape:", data.shape)
    print(f"{name} missing values:", np.isnan(data).sum())
    print(f"{name} infinite values:", np.isinf(data).sum())

# Save fitted preprocessing pipeline
output_dir = PROJECT_ROOT / "models"
output_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(preprocessor, output_dir / "preprocessor.joblib")

print("\nPreprocessing pipeline saved successfully.")

Training shape: (31647, 42)
Training missing values: 0
Training infinite values: 0
Validation shape: (6782, 42)
Validation missing values: 0
Validation infinite values: 0
Testing shape: (6782, 42)
Testing missing values: 0
Testing infinite values: 0

Preprocessing pipeline saved successfully.
